# Smart Grid Demand Forecasting: Exploratory Data Analysis

This notebook is a beginner-friendly companion to `src/run_eda.py`. The script is the single reproducible source of the calculations: it reads the validated master CSV, preserves all missing values, and writes concise tables, charts, and calculated findings under `results/eda/`.

All timestamps are kept in the source convention and labelled **UTC**. We do not interpret them as California local clock time. This notebook performs no modelling or train/test splitting.

## 1. Run the reproducible workflow

Running the script first ensures the notebook and command line produce the same artifacts. It never writes to the master CSV.

In [ ]:
from pathlib import Path
import runpy

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
runpy.run_path(ROOT / "src" / "run_eda.py", run_name="__main__")

## 2. Load concise result tables

Quality flags matter because a timestamp can exist even when its measurement does not. Demand analysis uses only `demand_data_complete=True`. Solar and wind analysis uses only `renewable_data_complete=True`. Residual demand and renewable share require both demand and renewable inputs to be complete. Missing measurements are never treated as zero.

In [ ]:
import pandas as pd

RESULTS = ROOT / "results" / "eda"
TABLES = RESULTS / "tables"
FIGURES = RESULTS / "figures"

validation = pd.read_csv(TABLES / "validation_summary.csv")
flags = pd.read_csv(TABLES / "quality_flag_counts.csv")
exclusions = pd.read_csv(TABLES / "analysis_exclusion_counts.csv")
print("Validation summary")
print(validation.to_string(index=False))
print("\nQuality-flag counts")
print(flags.to_string(index=False))
print("\nMetric-specific exclusions")
print(exclusions.to_string(index=False))

The validation tables confirm coverage, duplicate counts, continuity, and exactly how many rows each analysis excludes. The next cell shows only the documented problem timestamps rather than printing the full dataset.

In [ ]:
null_demand = pd.read_csv(TABLES / "null_demand_timestamps.csv")
null_renewables = pd.read_csv(TABLES / "null_renewable_value_timestamps.csv")
missing_block = pd.read_csv(TABLES / "missing_renewable_row_block.csv")
print("Five null-demand timestamps")
print(null_demand.to_string(index=False))
print("\nFive timestamps with null returned renewable values")
print(null_renewables.to_string(index=False))
print("\n24-hour block with missing SUN and WND source rows")
print(missing_block.to_string(index=False))

## 3. Descriptive statistics

Each row below uses only observations with the measurements needed for that metric. The `count` column makes the included sample size visible.

In [ ]:
statistics = pd.read_csv(TABLES / "descriptive_statistics.csv")
print(statistics.round(2).to_string(index=False))

## 4. Demand patterns

The daily chart shows short-term variation plus a trailing 7-day descriptive average. The hourly, weekday, heatmap, and annual-comparison charts show recurring group averages. Differences are descriptive; they do not prove what caused demand to change.

![Daily average demand with 7-day rolling average](../results/eda/figures/01_daily_average_demand_7_day_rolling.png)

![Average demand by hour](../results/eda/figures/02_average_demand_by_hour.png)

![Average demand by day of week](../results/eda/figures/03_average_demand_by_day_of_week.png)

![Monthly average and peak demand](../results/eda/figures/04_monthly_average_and_peak_demand.png)

![Hour and day demand heatmap](../results/eda/figures/05_demand_hour_day_heatmap.png)

![Demand comparison by year](../results/eda/figures/10_demand_comparison_by_year.png)

## 5. Solar, wind, renewable share, and residual demand

Solar and wind plots omit the 29 renewable-incomplete timestamps. Renewable share and residual demand also require valid demand. Residual demand here means observed demand minus observed solar-plus-wind generation; it is not a forecast.

![Solar and wind generation by hour](../results/eda/figures/06_solar_wind_generation_by_hour.png)

![Renewable share by hour](../results/eda/figures/07_renewable_share_by_hour.png)

![Residual demand by hour](../results/eda/figures/08_residual_demand_by_hour.png)

![Monthly renewable generation](../results/eda/figures/09_monthly_renewable_generation.png)

![Monthly renewable share](../results/eda/figures/11_monthly_renewable_share.png)

## 6. Peak-period tables

These short tables preserve the timestamp and relevant measurements. They help distinguish peak demand, peak residual demand, and peak renewable share, which need not occur together.

In [ ]:
for label, filename in [
    ("Top 10 demand timestamps", "top_10_demand_timestamps.csv"),
    ("Top 10 residual-demand timestamps", "top_10_residual_demand_timestamps.csv"),
    ("Top 10 renewable-share timestamps", "top_10_renewable_share_timestamps.csv"),
]:
    print(f"\n{label}")
    print(pd.read_csv(TABLES / filename).round(2).to_string(index=False))

## 7. Calculated findings and justified conclusions

The report below is generated from the same filtered tables. Its conclusions are limited to observed patterns. It makes no causal claims and flags missing and negative source measurements for later review.

In [ ]:
print((RESULTS / "eda_findings.md").read_text(encoding="utf-8"))